In [1]:
from numba import njit, prange
import numpy as np
import faiss

data = np.load("/home/dhem/workspace/2024.3/data/save/train_test-0-0-0-0.npz")

x = data["x"]
y = data["y"]
w = data["w"]
coor = data["coor"]
name = data["name"]

In [2]:
(num_sample, dim_sample) = x.shape
res = faiss.StandardGpuResources()
flat_config = faiss.GpuIndexFlatConfig()
flat_config.device = 0
index = faiss.GpuIndexFlatL2(res, dim_sample, flat_config)
index.add(x)

In [3]:
b = x.copy()
min_number = 4
distances, indices = index.search(b, min_number)

var_y = np.var(np.einsum("ij,i->ij", y[indices], x[:, 0] * w), axis=1)
argsort_ = np.argsort(var_y)[::-1][:100]
print(np.einsum("ij,i->ij", y[indices], x[:, 0] * w)[argsort_])

[[-2.34545562e-05 -5.49169466e-06 -5.54375475e-06 -3.11629277e-06]
 [-2.24499997e-05 -1.97269743e-05 -1.97739302e-05 -2.39411726e-06]
 [-3.57179707e-05 -3.50192479e-05 -2.75839811e-05 -1.79692158e-05]
 [-1.82766582e-05 -6.48059611e-06 -6.44310956e-06 -2.14753862e-05]
 [-2.77213205e-05 -2.82830714e-05 -2.82071761e-05 -1.35998062e-05]
 [-1.57575839e-05 -4.84203973e-06 -4.23506972e-06 -1.77646171e-05]
 [-1.28683944e-05 -1.46446914e-05 -1.56174204e-06 -1.50000238e-06]
 [-1.57497245e-05 -1.66985694e-05 -3.54176463e-06 -1.95410574e-05]
 [-1.46900977e-05 -1.67176831e-05 -3.55894943e-06 -3.55927694e-06]
 [-1.30772395e-05 -1.50731695e-05 -1.96957287e-06 -2.04656701e-06]
 [-2.72276519e-05 -2.88982111e-05 -2.11827097e-05 -1.35523133e-05]
 [-1.48112776e-05 -1.77543151e-05 -1.71302150e-05 -3.08652482e-06]
 [-1.55247225e-05 -1.67575911e-05 -4.30436124e-06 -4.31744391e-06]
 [-1.68680803e-05 -1.87906677e-05 -6.14765314e-06 -6.14790330e-06]
 [-1.30904461e-05 -1.48007494e-05 -1.50405105e-05 -1.57583496e

In [4]:
distance = np.sum(
    np.transpose(
        np.transpose(x[indices[argsort_]], axes=(0, 2, 1)) - x[argsort_][:, :, None],
        axes=(0, 2, 1),
    ) ** 2,
    axis=2,
)
# print(x[argsort_][:, :, None].shape)
# print(np.transpose(x[indices[argsort_]], axes=(0, 2, 1)).shape)
energy = np.einsum("ij,i->ij", y[indices[argsort_]], (x[:, 0] * w)[argsort_])
print(y[indices[argsort_]] - y[argsort_][:, None])
print(energy)

[[ 0.00000000e+00  1.38448136e-02  1.38046884e-02  1.56756464e-02]
 [ 0.00000000e+00  1.96006625e-03  1.92626686e-03  1.44364641e-02]
 [ 0.00000000e+00  4.13344997e-04  4.81184179e-03  1.04996693e-02]
 [ 0.00000000e+00  1.00141985e-02  1.00460224e-02 -2.71554154e-03]
 [ 0.00000000e+00 -4.01594315e-04 -3.47336908e-04  1.00954347e-02]
 [ 0.00000000e+00  1.10162043e-02  1.16287716e-02 -2.02554156e-03]
 [ 0.00000000e+00 -1.96006625e-03  1.24763978e-02  1.25445249e-02]
 [ 0.00000000e+00 -8.99322099e-04  1.15707936e-02 -3.59345299e-03]
 [ 0.00000000e+00 -2.03665208e-03  1.11809228e-02  1.11805938e-02]
 [ 0.00000000e+00 -2.16821779e-03  1.20664755e-02  1.19828353e-02]
 [ 0.00000000e+00 -1.17093994e-03  4.23706275e-03  9.58541301e-03]
 [ 0.00000000e+00 -2.97746615e-03 -2.34606511e-03  1.18619131e-02]
 [ 0.00000000e+00 -1.22187430e-03  1.11203020e-02  1.11073360e-02]
 [ 0.00000000e+00 -1.71834039e-03  9.58153746e-03  9.58131387e-03]
 [ 0.00000000e+00 -1.85964802e-03 -2.12034525e-03  1.25200737e

In [12]:
from matplotlib import pyplot as plt

color_dict = {
    "methane_cc-pVDZ_0-1_1_-0.2000": "#004D40",
    "methane_cc-pVDZ_0-1_1_-0.1000": "#1A237E",
    "methane_cc-pVDZ_0-1_1_0.0000": "#7B1FA2",
    "methane_cc-pVDZ_0-1_1_0.1000": "#B71C1C",
    "methane_cc-pVDZ_0-1_1_0.2000": "#FF6F00",
}

plt.rcParams["figure.figsize"] = np.array([3, 3]) * 520 / 72

f, axes = plt.subplots(10, 10)
axes = axes.reshape(10, 10)

begin_y = 0.025
end_y = 0.95
int_y = 0.0
begin_x = 0.025
end_x = 0.95
int_x = 0.0
end_x += int_x
end_y += int_y

shapexy = np.shape(axes)
inter_x = np.linspace(begin_x, end_x, shapexy[1] + 1)
inter_y = np.linspace(begin_y, end_y, shapexy[0] + 1)

delta_x = inter_x[1] - inter_x[0] - int_x
delta_y = inter_y[1] - inter_y[0] - int_y

for i in range(shapexy[0]):
    for j in range(shapexy[1]):
        axes[i][j].set_position(
            [
                inter_x[j],
                inter_y[i],
                inter_x[j + 1] - inter_x[j] - int_x,
                inter_y[i + 1] - inter_y[i] - int_y,
            ]
        )
        axes[i][j].xaxis.set_tick_params(
            direction="in", which="both", bottom=True, top=True
        )
        axes[i][j].yaxis.set_tick_params(
            direction="in", which="both", left=True, right=True
        )
        if i != 0:
            axes[i][j].set_xticks([])
        if j != 0:
            axes[i][j].set_yticks([])


for i in range(argsort_.shape[0]):
    axes_i, axes_j = np.unravel_index(i, (10, 10))
    for j in range(indices.shape[1]):
        axes[axes_i, axes_j].scatter(
            distance[i][j],
            np.abs(energy[i][j] - energy[i][0]) * 627.509,
            c=color_dict[name[indices[argsort_[i]]][j]],
        )
        axes[axes_i, axes_j].set_xlim(-0.0035, 0.033)
        axes[axes_i, axes_j].set_ylim(-0.003, 0.033)
plt.savefig("test.pdf", dpi=300)
plt.clf()

<Figure size 2166.67x2166.67 with 0 Axes>

In [20]:
np.exp(-0.001*10)

0.9900498337491681